Paraller Workflow:

<pre>
Input (Runs, Balls, Fours, Sixes)
                |
   +------------+-------------+
   |            |             |
Strike Rate  Balls/Boundary  Boundary %
   |            |             |
   +------------+-------------+
                |
             Summary
                |
              Output
</pre>


Here Strike rate, Balls Per boundary and Boundary percentage will be calculated parallelly rather than calculating in sequence becauce can be calculated individually then once this all are generated a summary is made ad output is displayed

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START,END
from typing import TypedDict
from dotenv import load_dotenv
load_dotenv()

In [ ]:
class Cricket(TypedDict):
    runs : int
    balls : int
    four : int
    six : int

    sr : float
    bpb : float
    bp : float
    summary: str

In [ ]:
def strike_rate(state:Cricket):

    sr = (state['runs']/state['balls'])*100

    return {'sr':sr}

In [ ]:
def balls_per_boundary(state:Cricket):

    bpb = state['balls']/(state['four']*state['six'])

    return {'bpb':bpb}

In [ ]:
def boundary_percentage(state:Cricket):

    bp = ((state['four']*4) + (state['six']*6))/(state['runs']*100)

    return {'bp':bp}

In [ ]:
def summary1(state:Cricket):

    summary = f"Player Summary:\n Balls:{state['balls']} \n Runs:{state['runs']} \n Four:{state['four']} \n Sixes:{state['balls']} \n Strike Rate:{state['sr']} \n Balls Per Boundary:{state['bpb']} Boundary Percentage:{state['bp']}"

    state['summary'] = summary

    return state

In [ ]:
graph = StateGraph(Cricket)

In [ ]:
graph.add_node('strike_rate',strike_rate)
graph.add_node('balls_per_boundary',balls_per_boundary)
graph.add_node('boundary_percentage',boundary_percentage)
graph.add_node('summary',summary1)

In [ ]:
graph.add_edge(START,'strike_rate')
graph.add_edge(START,'balls_per_boundary')
graph.add_edge(START,'boundary_percentage')

graph.add_edge('strike_rate','summary')
graph.add_edge('balls_per_boundary','summary')
graph.add_edge('boundary_percentage','summary')

graph.add_edge('summary',END)

In [ ]:
flow = graph.compile()

In [ ]:
flow

In [ ]:
input = {
    'runs': 100,
    'balls':50,
    'four':6,
    'six':4
}

flow.invoke(input)

In [ ]:
flow